# Modelagem da Camada Gold: Dimensão Local

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para importações locais
sys.path.append(str(Path.cwd().parent.parent))

from pyspark.sql import functions as F
from src.services.spark_session import get_spark_session, close_spark_session
import src.modules.modeling_dim_utils as modeling
import src.modules.utils as utils

In [ ]:
# Inicializa a SparkSession conectada ao cluster do container
spark = get_spark_session("ModelagemGoldDimLocal")

# Leitura das tabelas da camada Silver

In [ ]:
# Define caminhos das origens na Silver
silver_vendas_path = "s3a://silver/vendas"

# Lê os dados da Silver definindo como None caso a origem não exista
try:
    df_vendas = spark.read.parquet(silver_vendas_path)
except Exception as e:
    print(f"Aviso: Tabela Silver de Vendas não encontrada: {e}")
    df_vendas = None

# Colunas de Local na Origem

In [ ]:
# Mostra a pré-visualização das colunas caso a tabela não seja None
if df_vendas is not None:
    print("=== Colunas em Silver Vendas ===")
    display(df_vendas.select("cidade", "estado").distinct().limit(5).toPandas())

# Cria a Dimensão Local (dim_local)

In [ ]:
# Executa a lógica de modelagem unificada
df_dim_local = modeling.create_dim_local(df_vendas)

if df_dim_local is not None:
    # Adiciona a data de carga
    df_dim_local = df_dim_local.withColumn("data_carga", F.to_date(F.lit(utils.get_current_date_str())))
    
    # Exibe informações sobre o DataFrame gerado
    print(f"Quantidade total de locais únicos: {df_dim_local.count()}")
    df_dim_local.printSchema()
    display(df_dim_local.limit(10).toPandas())
else:
    print("Nenhum local foi processado (tabela Silver de Vendas estava ausente).")

In [ ]:
# Finaliza a sessão do Spark
close_spark_session(spark)